# randn-like-noise-source — worked example 3: VAE reparameterization trick preserves dtype across float32/float64

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `randn-like-noise-source`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

The VAE reparameterization trick computes `z = mu + sigma * eps` where `eps ~ N(0, I)`. Using `torch.randn_like(sigma)` ensures that `eps` has the same dtype and device as the VAE's latent statistics. This means the same `reparameterize(mu, sigma)` function works correctly whether the model is running in float32 or float64, without any manual dtype management.

## Worked solution

**Step 1 — The reparameterization function.** `reparameterize(mu, sigma)` takes the latent mean and (positive) standard deviation. It samples `eps = torch.randn_like(sigma)` — this single line handles dtype, device, and shape simultaneously.

**Step 2 — Compute z.** `z = mu + sigma * eps`. Because `eps` inherits dtype from `sigma`, and `mu` and `sigma` share a dtype, the result `z` is in the same dtype as the inputs.

**Step 3 — Test with float32 and float64.** We call the function with both dtypes and verify that `z.dtype` always matches the input dtype — the function is dtype-agnostic.

**Step 4 — Gradient flow check.** We verify that `z.requires_grad` is True when `mu.requires_grad` is True (assuming `mu` is a leaf that needs gradients), confirming the reparameterization trick keeps the computation graph connected.

In [ ]:
import torch as t

def reparameterize(mu: t.Tensor, sigma: t.Tensor) -> t.Tensor:
    """VAE reparameterization: z = mu + sigma * eps, eps ~ N(0, I)."""
    eps = t.randn_like(sigma)    # inherits shape, dtype, device from sigma
    return mu + sigma * eps

# Test with float32
t.manual_seed(7)
B, D = 8, 16
mu32 = t.zeros(B, D, dtype=t.float32)
sigma32 = t.ones(B, D, dtype=t.float32)
z32 = reparameterize(mu32, sigma32)
print(f'float32 z dtype: {z32.dtype}')   # torch.float32
print(f'float32 z shape: {z32.shape}')   # (8, 16)

# Test with float64
t.manual_seed(7)
mu64 = t.zeros(B, D, dtype=t.float64)
sigma64 = t.ones(B, D, dtype=t.float64)
z64 = reparameterize(mu64, sigma64)
print(f'float64 z dtype: {z64.dtype}')   # torch.float64
print(f'float64 z shape: {z64.shape}')   # (8, 16)

# Gradient flow: requires_grad propagates through the trick
mu_leaf = t.zeros(4, 8, requires_grad=True)
sigma_leaf = t.ones(4, 8)
z_grad = reparameterize(mu_leaf, sigma_leaf)
print(f'z requires_grad: {z_grad.requires_grad}')  # True